# Python 101 - Solutions
## Chapter VI. extra

---

**For teaching assistants.** This notebook mirrors the exercises in
`../python101_06_extra.ipynb` one for one. Most solutions end with an `assert`, so
running the whole notebook top to bottom is also a self-test: if it runs clean,
every solution still works.

There is usually more than one right answer - if a student's version passes the
same `assert`, it is correct.

In [ ]:
# Run from the chapter folder, so that `helpers`, `./data/...` and `./pics/...`
# resolve exactly the way they do in the lecture notebooks.
import os
import sys

if os.path.basename(os.getcwd()) == 'solutions':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

print('working directory:', os.getcwd())

> **Note.** Like chapter VI, this one is currently skipped. Solutions here for completeness.

In [ ]:
import json
import os
import random
from collections import Counter

### 1. `GuessANumber` with error handling

The notebook's driver cell feeds it `['valami', 0.3, 5, 'a']` - a string, a float, a valid int and a letter - **and** uses the same list as the `limit` argument. So the constructor has to survive a nonsense limit too, not just `guess`. That is easy to miss.

In [ ]:
class GuessANumber:
    """Guess-the-number, which refuses to crash on bad input."""

    DEFAULT_LIMIT = 10

    def __init__(self, limit=10):
        if isinstance(limit, bool) or not isinstance(limit, int) or limit < 1:
            print(f'  (bad limit {limit!r}, falling back to {self.DEFAULT_LIMIT})')
            limit = self.DEFAULT_LIMIT
        self.limit = limit
        self.number = random.randint(1, limit)

    def guess(self, guess):
        """Return 'Win', 'Lower' or 'Higher'; complain about anything else."""
        try:
            value = int(guess)
        except (TypeError, ValueError):
            print(f"  '{guess}' is not a whole number!")
            return None

        if not 1 <= value <= self.limit:
            print(f'  {value} is outside 1..{self.limit}!')
            return None

        if value == self.number:
            return 'Win'
        return 'Lower' if value > self.number else 'Higher'


inputs = ['valami', 0.3, 5, 'a']
for inp in inputs:
    print('game init...', end='')
    game = GuessANumber(inp)
    print('done. Target:', game.number)
    for guess in inputs:
        print('- guess:', guess, '->', game.guess(guess))
    print('-' * 30)

game = GuessANumber(10)
game.number = 5
assert game.guess(5) == 'Win'
assert game.guess(7) == 'Lower' and game.guess(3) == 'Higher'
assert game.guess('nonsense') is None
assert game.guess(999) is None
assert GuessANumber('valami').limit == 10      # bad limit survived

### 2. `RPS` with error handling

In [ ]:
class RPS:
    """Rock-paper-scissors that rejects invalid moves instead of crashing."""

    trumps = {'r': 'p', 'p': 's', 's': 'r'}

    def __init__(self):
        self.hands = ['r', 'p', 's']
        self.ai = None

    def move(self):
        return random.choice(self.hands)

    def play(self, hand):
        # a plain `self.trumps[hand]` would raise KeyError or TypeError here
        try:
            counter = self.trumps[hand]
        except (KeyError, TypeError):
            print(f'  {hand!r} is not a valid move - pick one of {sorted(self.trumps)}')
            return None

        self.ai = self.move()
        self.hands.append(counter)

        if self.ai == hand:
            return 'draw'
        return 'win' if self.trumps[self.ai] == hand else 'lose'


inputs = ['valami', 0.3, 5, 'r']
rps = RPS()
for inp in inputs:
    print(inp, '->', rps.play(inp))

assert rps.play('valami') is None
assert rps.play(0.3) is None       # unhashable/invalid types handled too
assert rps.play('r') in ('win', 'lose', 'draw')

### 3. Count the files in a directory, by extension

`os.path.splitext` is cleaner than slicing on the last dot, and `Counter` does the tallying.

In [ ]:
def count_by_extension(directory='.'):
    counts = Counter()
    for entry in os.listdir(directory):
        path = os.path.join(directory, entry)
        if not os.path.isfile(path):
            continue
        extension = os.path.splitext(entry)[1].lstrip('.').lower()
        counts[extension or '(none)'] += 1
    return dict(counts)


here = count_by_extension('.')
print(here)

assert here.get('py', 0) >= 1          # helpers.py
assert here.get('ipynb', 0) >= 10      # the lecture notebooks
assert count_by_extension('./pics/').get('png', 0) > 5

### 4. A basic calculator

A dictionary of operations beats a chain of `if`s, and `~` has to be special-cased because it ignores its second argument.

In [ ]:
import operator


def calculate(first, second, operation):
    """Apply `operation` to two numbers. Returns None on any bad input."""
    operations = {
        '+': operator.add, '-': operator.sub,
        '*': operator.mul, '/': operator.truediv,
        '^': operator.pow,
        '&': lambda a, b: bool(a) and bool(b),
        '|': lambda a, b: bool(a) or bool(b),
        '=': operator.eq,
    }

    if operation == '~':
        try:
            return not bool(first)         # the second number is ignored
        except Exception as error:
            print('ERROR:', error)
            return None

    if operation not in operations:
        print(f'ERROR: unknown operation {operation!r}')
        return None

    try:
        return operations[operation](first, second)
    except ZeroDivisionError:
        print('ERROR: cannot divide by zero')
        return None
    except TypeError as error:
        print(f'ERROR: {error}')
        return None


for a, b, op in [(6, 3, '+'), (6, 3, '/'), (6, 0, '/'), (2, 10, '^'),
                 (1, 0, '&'), (1, 0, '|'), (2, 2, '='), (1, 99, '~'),
                 (6, '3', '*'), (6, '3', '/'), (6, 3, '%')]:
    print(f'  {a} {op} {b} = {calculate(a, b, op)}')

assert calculate(6, 3, '+') == 9
assert calculate(6, 3, '/') == 2.0
assert calculate(6, 0, '/') is None
assert calculate(2, 10, '^') == 1024
assert calculate(1, 0, '&') is False and calculate(1, 0, '|') is True
assert calculate(2, 2, '=') is True
assert calculate(1, 99, '~') is False      # second number ignored
assert calculate(6, 3, '%') is None

# A genuine surprise worth showing: `6 * '3'` is *valid* python - it repeats the
# string six times. So the calculator happily returns '333333' instead of
# raising TypeError. `6 / '3'` does raise, though.
assert calculate(6, '3', '*') == '333333'
assert calculate(6, '3', '/') is None
print("\n6 * '3' =", repr(calculate(6, '3', '*')), '<- not a TypeError!')

### 5. `SaveLoader`

The spec is fussy, so read it twice:
- saving **appends** (one json object per line), it must not overwrite
- loading returns **all** the saves as a list, then **deletes** the file
- loading a file that does not exist must not blow up

The expected result in the notebook is a *list of two dicts*, which is what tells you the format is one json object per line rather than one big object.

In [ ]:
class SaveLoader:
    """Saves variables to a file as json, one save per line."""

    def save(self, filename, **kwargs):
        # 'a' rather than 'w', so earlier saves survive
        with open(filename, 'a', encoding='utf-8') as handle:
            handle.write(json.dumps(kwargs) + '\n')
        return filename

    def load(self, filename):
        try:
            with open(filename, encoding='utf-8') as handle:
                saves = [json.loads(line) for line in handle if line.strip()]
        except FileNotFoundError:
            print(f'No save file called {filename!r}.')
            return None
        except json.JSONDecodeError as error:
            print(f'{filename!r} is corrupt: {error}')
            return None

        os.remove(filename)
        return saves


sl = SaveLoader()
sl.save('mylittlesave.sav', a=7, b=8, c={'a': 7, 'b': 8}, d=[])

a, b = 7, 8
c = {'a': a, 'b': b}
d = [a, b, c]
sl.save('mylittlesave.sav', d=d)

assert sl.load('mylittlenonexistingsave.sav') is None

loaded = sl.load('mylittlesave.sav')
print(loaded)

assert loaded == [{'a': 7, 'b': 8, 'c': {'a': 7, 'b': 8}, 'd': []},
                  {'d': [7, 8, {'a': 7, 'b': 8}]}]
assert not os.path.exists('mylittlesave.sav')    # load removes the file
print('\nmatches the expected result in the notebook exactly')

### Custom exceptions

The notebook defines `FuliError` and its subclasses but never uses them. Here is what they are for: one `except` for the whole family.

In [ ]:
class FuliError(Exception):
    """Base class for our exceptions."""


class NumberError(FuliError):
    def __init__(self, number, explanation):
        super().__init__(explanation)
        self.number = number
        self.exp = explanation


class CharacterError(FuliError):
    def __init__(self, character):
        super().__init__('You messed with the wrong character, buddy!')
        self.character = character
        self.exp = 'You messed with the wrong character, buddy!'


def validate(value):
    if isinstance(value, str):
        raise CharacterError(value)
    if value < 0:
        raise NumberError(value, 'negative numbers are not allowed')
    return value


for value in (5, -1, 'x'):
    try:
        print(value, '->', validate(value))
    except FuliError as error:          # one except for the whole family
        print(f'{value!r} -> {type(error).__name__}: {error.exp}')

assert validate(5) == 5
for bad, expected in ((-1, NumberError), ('x', CharacterError)):
    try:
        validate(bad); raise AssertionError
    except FuliError as error:
        assert isinstance(error, expected)
        assert isinstance(error, FuliError)